# Reproduce `submission_v84.csv` (public LB 0.39, OOF PQ 0.459)

End-to-end inference from the three committed weight files in `weights/`. No training, no test labels, no external data.

| component | file | sha256 |
|---|---|---|
| detector: YOLO11s-seg "p2first" (P2 stride-2 mask prototypes), all-data fit | `weights/p2first_detector.pt` | `7ff39296…` |
| crop refiner: ResNet18 U-Net (`CropUNet`, 256 px crops), V84 soft-target recipe, all-data fit | `weights/soft_all_refiner_state.pt` | `ef9e3810…` |
| demote rule: LightGBM on 28 per-candidate features, λ = 0.458722 | `weights/demote_all.txt` + `demote_all.json` | `4b8c2375…` / `6c8a7863…` |

Pipeline per test image: detector at 1024 px with 4-flip consensus (conf 0.30) → crop refiner at threshold 0.50 → drop masks < 200 px → demote rule → non-overlapping COCO RLE.
Reference output: 1112 rows over 177 of 180 images, file sha256 `2925e2a2…`, canonical (row-sorted) sha256 `c5df205e…`.

Runtime: ~25 min on one T4-class GPU for all 180 images; set `SMOKE = 3` for a 2-minute check.

In [ ]:
import os, sys, json, hashlib, subprocess, csv
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "scripts/export_v74.py").exists() else Path.cwd().parent
assert (ROOT / "scripts/export_v74.py").exists(), "run from the repository root or notebooks/"
os.chdir(ROOT)

# Test images: competition layout <DATA_ROOT>/test/test_images/*.jpeg
CANDIDATES = [os.environ.get("SOLAR_DATA_ROOT"),
              "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026",
              "/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026",
              str(ROOT / "data/raw/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")]
DATA_ROOT = next(Path(p) for p in CANDIDATES if p and (Path(p) / "test/test_images").is_dir())
os.environ["SOLAR_DATA_ROOT"] = str(DATA_ROOT)

DEVICE = os.environ.get("SOLAR_DEVICE", "cuda:0")   # "cpu" works, ~40x slower
SMOKE = int(os.environ.get("SOLAR_SMOKE", "0"))     # >0: only the first N images (no reference match expected)
OUT = ROOT / "outputs/reproduce_v84"
PY = sys.executable
print("root", ROOT, "\ndata", DATA_ROOT, "\ndevice", DEVICE, "smoke", SMOKE)
print("test images:", len(list((DATA_ROOT / "test/test_images").glob("*.jpeg"))))

In [ ]:
# 1. Weight identity
def sha256(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
W = ROOT / "weights"
expected = dict(line.split()[::-1] for line in (W / "SHA256SUMS").read_text().splitlines() if line.strip())
for name, want in expected.items():
    got = sha256(W / name)
    assert got == want, f"{name}: {got} != {want}"
    print(f"ok  {name:30s} {got[:16]}…")
DET, REF, DEMOTE = W / "p2first_detector.pt", W / "soft_all_refiner_state.pt", W / "demote_all.json"

In [ ]:
# 2. Inference: one shard over all test images (the original run used 3 GPU shards; sharding only changes row order)
(OUT / "export").mkdir(parents=True, exist_ok=True)
cmd = [PY, "scripts/export_v74.py", "--shard-index", "0", "--num-shards", "1",
       "--detector", str(DET), "--detector-sha", sha256(DET),
       "--refiner", str(REF), "--refiner-sha", sha256(REF),
       "--demote-model", str(DEMOTE),
       "--imgsz", "1024", "--mask-threshold", "0.50", "--confidence", "0.30", "--min-area", "200",
       "--device", DEVICE, "--output-dir", str(OUT / "export")]
if SMOKE: cmd += ["--limit", str(SMOKE)]
print(" ".join(cmd))
env = dict(os.environ)
if DEVICE.startswith("cuda") and "CUDA_VISIBLE_DEVICES" not in env:
    env["CUDA_VISIBLE_DEVICES"] = DEVICE.split(":")[-1]; cmd[cmd.index("--device") + 1] = "cuda:0"
with open(OUT / "export.log", "w") as log:
    proc = subprocess.run(cmd, env=env, stdout=log, stderr=subprocess.STDOUT, text=True)
print((OUT / "export.log").read_text()[-1500:])
assert proc.returncode == 0, "export failed, see outputs/reproduce_v84/export.log"

In [ ]:
# 3. Assemble, validate, compare with the submitted file
import pandas as pd
sys.path.insert(0, str(ROOT / "scripts"))
from validate_submission import validate_submission

manifest = json.loads((OUT / "export/shard_0.json").read_text())
assert manifest["status"] == ("partial" if SMOKE else "complete"), manifest["status"]
df = pd.read_csv(OUT / "export/shard_0.csv", dtype=str)
assert df.filament_id.is_unique
sub = OUT / "submission_v84_reproduced.csv"
df.to_csv(sub, index=False)

stems = {p.stem for p in (DATA_ROOT / "test/test_images").glob("*.jpeg")}
if not SMOKE:
    report = validate_submission(sub, stems, min_area=200)
    print(json.dumps({k: v for k, v in report.items() if k != "per_image"}, indent=2)[:1500])

def canonical_sha(path):
    rows = sorted(tuple(r) for r in csv.reader(open(path)))
    h = hashlib.sha256()
    for r in rows: h.update(("\t".join(r) + "\n").encode())
    return h.hexdigest(), len(rows) - 1

REFERENCE = "c5df205e8303321cb426f5621a903748bd301c921085f80b674c6a9c08134edb"  # row-sorted sha256 of submission_v84.csv (1112 rows)
got, n = canonical_sha(sub)
print(f"rows {n}  canonical sha256 {got}")
print("file sha256", sha256(sub))
if SMOKE:
    print("smoke run: reference comparison skipped")
else:
    print("MATCHES submitted V84" if got == REFERENCE else "DIFFERS from submitted V84 (GPU nondeterminism can flip a few border pixels; compare row counts and per-image coverage)")

## Notes

- `scripts/export_v74.py` is the exact export used for the submission (V74 export path + V84 `--demote-model`); its per-image logic is `refine_image()`.
- Training recipes (not needed to reproduce): detector `scripts/train_v61_fold.py` (p2first config, see `docs/v61*`), refiner `scripts/train_v82_refiner_targets.py --arm soft --train-all`, demote `scripts/v84_demote_fit.py`. Full ledger: `docs/v84_export_2026-10-11.md`, `docs/v84_polish_bundle_result_2026-10-11.md`.
- The repo copy `kaggle_inference/solar_base.py` (CropUNet definition) is byte-identical to the one the training runs imported.